In [ ]:
import requests
import os
import zipfile
import pandas as pd
import glob
import duckdb
from tqdm import tqdm



In [ ]:
os.makedirs('dados_bolsa_familia', exist_ok=True)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Referer": "https://www.portaldatransparencia.gov.br/",
}

downloads = {
    2017: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201701",
    2018: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201801",
    2019: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201901",
    2020: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/202001",
    2021: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/202101",
    2023: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202303",
    2024: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202401",
    2025: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202501",
    2026: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202601",
}

for ano, url in downloads.items():
    print(f"Baixando {ano}...", end=" ")
    try:
        response = requests.get(url, headers=headers, timeout=120)
        if response.status_code == 200:
            zip_path = f"dados_bolsa_familia/{ano}.zip"
            with open(zip_path, 'wb') as f:
                f.write(response.content)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall('dados_bolsa_familia/')
            os.remove(zip_path)
            print("OK")
        else:
            print(f"Erro HTTP {response.status_code}")
    except Exception as e:
        print(f"Falhou: {e}")

print("\nPronto! Arquivos em ./dados_bolsa_familia/")

In [ ]:
#Estabelecer conexão
con = duckdb.connect('dados_bolsa_familia/unificado.duckdb')

In [ ]:
con.execute("DROP TABLE IF EXISTS bolsa_familia")

con.execute("""
    CREATE TABLE bolsa_familia AS
    SELECT * FROM read_csv(
        'dados_bolsa_familia/*.csv',
        delim=';',
        header=true,
        encoding='latin-1',
        ignore_errors=true
    )
""")

total = con.execute("SELECT COUNT(*) FROM bolsa_familia").fetchone()[0]
print(f"Total de registros carregados: {total:,}")

In [13]:
#Ranking de recebimento do bolsa familia
top_nis = con.execute("""
    WITH por_ano AS (
        SELECT
            "NIS FAVORECIDO",
            CAST(LEFT(CAST("MÊS COMPETÊNCIA" AS VARCHAR), 4) AS INT) AS ano,
            MAX(CAST(REPLACE("VALOR PARCELA", ',', '.') AS DOUBLE)) AS valor_ano
        FROM bolsa_familia
        WHERE "CPF FAVORECIDO" IS NOT NULL
        GROUP BY "NIS FAVORECIDO", ano
    )
    SELECT
        "NIS FAVORECIDO",
        SUM(CASE WHEN ano != 2026 THEN valor_ano * 12 ELSE valor_ano END) AS total_estimado
    FROM por_ano
    GROUP BY "NIS FAVORECIDO"
    ORDER BY total_estimado DESC
    LIMIT 500
""").df()["NIS FAVORECIDO"].tolist()


nis_lista = ", ".join([f"'{n}'" for n in top_nis])

#Utiliza o valor de cada ano em que a pessoa aparece e multiplica por 12
resultado = con.execute(f"""
    WITH base AS (
        SELECT
            "NIS FAVORECIDO", "NOME FAVORECIDO", "CPF FAVORECIDO",
            "UF", "NOME MUNICÍPIO",
            CAST(LEFT(CAST("MÊS COMPETÊNCIA" AS VARCHAR), 4) AS INT) AS ano,
            MAX(CAST(REPLACE("VALOR PARCELA", ',', '.') AS DOUBLE)) AS valor_ano
        FROM bolsa_familia
        WHERE "CPF FAVORECIDO" IS NOT NULL
          AND "NIS FAVORECIDO" IN ({nis_lista})
        GROUP BY "NIS FAVORECIDO", "NOME FAVORECIDO", "CPF FAVORECIDO", "UF", "NOME MUNICÍPIO", ano
    ),
    estimado AS (
        SELECT
            "NOME FAVORECIDO", "NIS FAVORECIDO", "CPF FAVORECIDO",
            MAX("UF") AS uf,
            MAX("NOME MUNICÍPIO") AS municipio,
            MIN(ano) AS primeiro_recebimento,
            MAX(ano) AS ultimo_recebimento,
            COUNT(DISTINCT ano) AS qtd_anos,
            SUM(CASE WHEN ano != 2026 THEN valor_ano * 12 ELSE valor_ano END) AS total_estimado
        FROM base
        GROUP BY "NOME FAVORECIDO", "NIS FAVORECIDO", "CPF FAVORECIDO"
    )
    SELECT * FROM estimado
    ORDER BY total_estimado DESC
    LIMIT 20
""").df()

print("\n=== TOP 20 MAIORES BENEFICIÁRIOS (ESTIMATIVA TOTAL) ===\n")
print(resultado.to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== TOP 20 MAIORES BENEFICIÁRIOS (ESTIMATIVA TOTAL) ===

                      NOME FAVORECIDO  NIS FAVORECIDO  CPF FAVORECIDO  uf             municipio  primeiro_recebimento  ultimo_recebimento  qtd_anos  total_estimado
0              LUCIA MARIA DOS SANTOS     16436638270  ***.173.375-**  BA                ITIUBA                  2017                2026         9        183972.0
1                  JOAO DE DEUS SIRVO     23653255100  ***.151.471-**  MT             QUERENCIA                  2017                2026         9        170026.0
2   MARIA DO LIVRAMENTO DANTAS SOARES     16484887331  ***.210.322-**  AM                BERURI                  2017                2026         9        167376.0
3           MARIA LUIZA SOARES SANTOS     16484426997  ***.134.974-**  PE           AGUAS BELAS                  2017                2026         9        167034.0
4               DORA ARNETE RODRIGUES     16105187832  ***.381.392-**  AC  MARECHAL THAUMATURGO                  2017     